# Sprint 3 — Representation & Clustering (Colab launcher)

This notebook only **launches** `sprint3_colab_experiment.py` and **reads** its outputs. The script is the source of
truth; no experiment logic lives here and nothing is recalculated in this notebook.

Controlled 2 × 2 experiment on the frozen input `sprint3_task_input.csv` (3,015 task statements):

1. **E5 + K-Means** (`intfloat/e5-base-v2`)
2. **E5 + HDBSCAN**
3. **Sentence-T5 + K-Means** (`sentence-transformers/sentence-t5-base`)
4. **Sentence-T5 + HDBSCAN**

**Before running:** upload `Sprint3_Experiment_Colab.ipynb`, `sprint3_colab_experiment.py` and `sprint3_task_input.csv`
to **My Drive/CAPESTONE PROJECT/**, then select *Runtime → Change runtime type → GPU*. Run the cells top to bottom.
If the runtime disconnects, re-run cells 3–6: the script resumes from its checkpoints.

In [ ]:
# CELL 2 — check GPU
!nvidia-smi
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU - the experiment still runs on CPU, only slower (Runtime > Change runtime type > GPU).")

In [ ]:
# CELL 3 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 4 — paths (verified before continuing)
import os

PROJECT_DIR = "/content/drive/MyDrive/CAPESTONE PROJECT"
SCRIPT = f"{PROJECT_DIR}/sprint3_colab_experiment.py"
INPUT = f"{PROJECT_DIR}/sprint3_task_input.csv"
OUTPUT = f"{PROJECT_DIR}/sprint3_results"
ZIP = f"{PROJECT_DIR}/Sprint3_Experiment_Results.zip"

for name, path in [("SCRIPT", SCRIPT), ("INPUT", INPUT), ("OUTPUT", OUTPUT), ("ZIP", ZIP)]:
    print(f"{name:7s} {path}")

missing = [p for p in (SCRIPT, INPUT) if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError("Upload these files to My Drive/CAPESTONE PROJECT/ first: " + ", ".join(missing))
print("\nSCRIPT and INPUT found.")

In [ ]:
# CELL 5 — install dependencies (exactly the packages sprint3_colab_experiment.py imports;
# packages Colab already provides are left as they are)
!pip install -q numpy pandas scipy scikit-learn matplotlib torch transformers sentence-transformers sentencepiece hdbscan umap-learn

In [ ]:
# CELL 6 — run the complete experiment
!python "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_colab_experiment.py" \
  --input "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_task_input.csv" \
  --output "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results"

In [ ]:
# CELL 7 — verify outputs (read-only)
import json, os
from pathlib import Path
from IPython.display import Markdown, display

if os.path.isfile(ZIP):
    print(f"ZIP: {ZIP}\nsize: {os.path.getsize(ZIP) / 1e6:.1f} MB")
else:
    print(f"ZIP NOT FOUND: {ZIP} - check the output of cell 6 for errors.")

out = Path(OUTPUT)
print("\nOutput directories:")
for d in ["embeddings", "evaluation", "final", "figures", "input"]:
    p = out / d
    n = len([f for f in p.rglob("*") if f.is_file()]) if p.exists() else 0
    print(f"  {d:11s} {'OK ' if p.exists() else 'MISSING'} {n} files")

manifest = None
mf = out / "run_manifest.json"
if mf.exists():
    manifest = json.loads(mf.read_text())
    ch = manifest.get("chosen_configuration", {})
    print("\nWinning pipeline (from run_manifest.json):", ch.get("config_id"),
          "|", ch.get("embedding_model"), "|", ch.get("clustering_method"), "|", ch.get("parameters"))

ms = out / "evaluation" / "model_selection.md"
if ms.exists():
    display(Markdown(ms.read_text()))
else:
    print("model_selection.md not found")

In [ ]:
# CELL 8 — optional result preview (reads the generated files; recalculates nothing)
import pandas as pd

cmp_path = out / "evaluation" / "experiment_comparison.csv"
if cmp_path.exists():
    cmp = pd.read_csv(cmp_path)
    display(cmp)
    def show(label, row):
        print(f"{label}: {row['config_id']}")
        print(f"  final leaf topics:      {row['n_clusters']}")
        print(f"  noise share (pre-rule): {row['noise_share']}")
        print(f"  semantic cohesion:      {row['semantic_cohesion']} (lift {row['semantic_cohesion_lift']})")
        print(f"  stability (subsample ARI): {row['stability_ari']} +/- {row['stability_ari_sd']}")
    for label, tag in [("WINNING PIPELINE", "1 (winner)"), ("SECOND-BEST PIPELINE", "2 (second)")]:
        hit = cmp[cmp["final_rank"].astype(str) == tag]
        if len(hit):
            show(label, hit.iloc[0])
else:
    print("experiment_comparison.csv not found")

mem_path = out / "final" / "task_topic_membership.csv"
top_path = out / "final" / "task_topics.csv"
if top_path.exists():
    print("\nfinal leaf topics in task_topics.csv:", len(pd.read_csv(top_path)))
if mem_path.exists():
    mem = pd.read_csv(mem_path)
    print("UNASSIGNED / noise statements in task_topic_membership.csv:", int((mem["topic_id"] == "UNASSIGNED").sum()))